In [ ]:
# | Rank | Model                      | Verdict                                            |
# | ---- | -------------------------- | -------------------------------------------------- |
# | 🥇 1 | **BAAI/bge-base-en-v1.5**  | Best overall                                       |
# | 🥈 2 | **intfloat/e5-large-v2**   | Very close second                                  |
# | 🥉 3 | **BAAI/bge-large-en-v1.5** | Similar to base but slightly worse retrieval order |
# | 4    | **BAAI/bge-m3**            | Mixed Reg 39 and 40 badly                          |
# | 5    | **all-MiniLM-L6-v2**       | Weakest                                            |


In [ ]:
# EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

In [ ]:
import fitz
import faiss
import numpy as np
import ollama

from sentence_transformers import SentenceTransformer


# =====================================================
# CONFIG
# =====================================================

PDF_PATH = "/Users/admin/Downloads/1767338915207_1.pdf"

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# Alternatives:
# sentence-transformers/all-MiniLM-L6-v2
# BAAI/bge-small-en-v1.5
# BAAI/bge-base-en-v1.5
# BAAI/bge-large-en-v1.5
# BAAI/bge-m3

LLM_MODEL = "mistral:latest"

TOP_K = 15


# =====================================================
# PDF
# =====================================================

def extract_pdf_pages(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc):

        text = page.get_text("text")

        if text.strip():

            pages.append(
                {
                    "page": page_num + 1,
                    "text": text
                }
            )

    return pages


# =====================================================
# EMBEDDINGS
# =====================================================

print(f"\nLoading {EMBEDDING_MODEL}")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Loaded.")


def create_embeddings(pages):

    texts = [p["text"] for p in pages]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=16
    )

    return np.array(embeddings).astype("float32")


# =====================================================
# INDEX
# =====================================================

def build_index(embeddings):

    dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)

    index.add(embeddings)

    return index


# =====================================================
# RETRIEVE
# =====================================================

def retrieve(question, index, pages):

    q_emb = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    q_emb = np.array(q_emb).astype("float32")

    scores, ids = index.search(
        q_emb,
        TOP_K
    )

    results = []

    for score, idx in zip(scores[0], ids[0]):

        results.append(
            {
                "score": float(score),
                "page": pages[idx]["page"],
                "text": pages[idx]["text"]
            }
        )

    return results


# =====================================================
# LLM
# =====================================================

def answer_question(question, retrieved_pages):

    context = "\n\n".join(
        [
            f"""
PAGE {item['page']}

{item['text']}
"""
            for item in retrieved_pages
        ]
    )

    prompt = f"""
You are a SEBI regulatory expert.

Answer ONLY from the supplied context.

If the question refers to Regulation 39,
only discuss Regulation 39.

If the question refers to Regulation 40,
only discuss Regulation 40.

Do not mix regulations.

For amendment questions explain:

1. Existing position
2. Problem identified
3. Why amendment was proposed
4. Public consultation feedback
5. Board decision
6. Any Board modifications

Context:

{context}

Question:

{question}

Answer:
"""

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


# =====================================================
# MAIN
# =====================================================

print("\nReading PDF")

pages = extract_pdf_pages(
    PDF_PATH
)

print(
    f"Pages indexed: {len(pages)}"
)

embeddings = create_embeddings(
    pages
)

index = build_index(
    embeddings
)

print("\nReady.\n")

while True:

    question = input(
        "\nQuestion: "
    )

    if question.lower() == "exit":
        break

    retrieved = retrieve(
        question,
        index,
        pages
    )

    print("\n")
    print("=" * 100)
    print("RETRIEVED PAGES")
    print("=" * 100)

    for r in retrieved[:10]:

        print(
            f"Page {r['page']} "
            f"| Score={r['score']:.4f}"
        )

    print("\n")
    print("=" * 100)
    print("ANSWER")
    print("=" * 100)

    answer = answer_question(
        question,
        retrieved
    )

    print(answer)


Loading BAAI/bge-base-en-v1.5
Loaded.

Reading PDF
Pages indexed: 70


Batches: 100%|██████████| 5/5 [00:13<00:00,  2.64s/it]


Ready.





RETRIEVED PAGES
Page 38 | Score=0.6393
Page 67 | Score=0.6310
Page 41 | Score=0.6295
Page 60 | Score=0.6295
Page 39 | Score=0.6194
Page 50 | Score=0.6150
Page 40 | Score=0.6146
Page 44 | Score=0.6136
Page 51 | Score=0.6116
Page 34 | Score=0.6111


ANSWER
 The provided information does not explicitly state why SEBI proposed an amendment to REG 61A. However, based on the context of the provided excerpts, it seems that the amendments are part of a broader review and update of regulations related to corporate governance for High Value Debenture (HVD) companies. The reasons for specific amendments may be addressed within the detailed explanations provided in the consultation paper or other official documents, but they are not explicitly stated in the excerpts provided.


RETRIEVED PAGES
Page 64 | Score=0.4988
Page 63 | Score=0.4972
Page 52 | Score=0.4961
Page 36 | Score=0.4933
Page 43 | Score=0.4906
Page 6 | Score=0.4903
Page 8 | Score=0.4901
Page 48 | Score=0.4897
Page 39 | Score=0.4884


KeyboardInterrupt: 